In [15]:
import os
import pefile
import csv
import random

<b>List of NOP-equivalent opcodes

In [16]:
NOP_EQUIVALENTS = [
    b'\x83\xC0\x00',  # ADD EAX, 0
    b'\x83\xE8\x00',  # SUB EAX, 0
    b'\x6B\xC0\x01',  # IMUL EAX, EAX, 1
    b'\x8D\x40\x00',  # LEA EAX, [EAX + 0]
    b'\x0B\xC0',      # OR EAX, EAX
    b'\x23\xC0',      # AND EAX, EAX
    b'\xD9\xD0'       # FNOP
    b'\x89\xC0'       # MOV EAX EAX
    b'\x87\xC0'       # XCHG EAX EAX
]

<b>Calculate the amount of slack space at the end of the section

In [17]:
def get_slack_space(section_data):
    zero_count = 0
    for byte in reversed(section_data):
        if byte == 0x00:
            zero_count += 1
        else:
            break
    return zero_count

<b> Inject random NOP-equivalent code into the slack space of a PE file

In [18]:
def inject_random_nop(file_path, nop_percentage=0.05, nop_code=None):
    pe = pefile.PE(file_path)
    for section in pe.sections:
        if section.Name.decode(errors='ignore').strip('\x00') == '.text':
            section_name = section.Name.decode(errors='ignore').strip('\x00')
            target_section = section
    section_data = target_section.get_data()
    zero_count = get_slack_space(section_data)

    if zero_count == 0:
        raise ValueError(f"No contiguous zero bytes found in section {section_name}")

    injection_size = int(zero_count * nop_percentage)
    if injection_size == 0:
        raise ValueError(f"Injection size is zero for {nop_percentage*100}% of slack space")

    if nop_code is None:
        nop_code = random.choice(NOP_EQUIVALENTS)
    nop_code_size = len(nop_code)

    if injection_size < nop_code_size:
        raise ValueError(f"Not enough slack space in {section_name} to inject {nop_code_size} byte NOP code")

    last_free_space_offset = len(section_data) - zero_count
    file_offset = target_section.PointerToRawData + last_free_space_offset

    with open(file_path, 'rb') as f:
        original_data = f.read()

    modified_data = (
        original_data[:file_offset] +
        nop_code +
        original_data[file_offset + nop_code_size:]
    )

    return modified_data, section_name, nop_code.hex()

<b>Process executables in the input folder, injecting NOP-equivalent code and logging results

In [19]:
def process_executables(input_folder, output_folder_base, csv_path, start_percentage=0.05, step=0.10, iterations=5):
    if not os.path.exists(output_folder_base):
        os.makedirs(output_folder_base)

    # Read existing CSV entries
    processed_files = {}
    if os.path.exists(csv_path):
        with open(csv_path, mode='r') as csv_file:
            csv_reader = csv.reader(csv_file)
            next(csv_reader)  # Skip header
            for row in csv_reader:
                key = (row[0], row[1], row[2])
                processed_files[key] = (row[3], row[4])

    with open(csv_path, mode='a', newline='') as csv_file:
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow(["File Name", "Modified Section", "NOP Code Inserted", "Status", "Error Message"])

        for root, _, files in os.walk(input_folder):
            for file_name in files:
                if file_name.endswith(".exe"):
                    file_path = os.path.join(root, file_name)
                    family_name = os.path.basename(root)
                    
                    # Create base folder for each percentage and family
                    for i in range(5, 50, 10):
                        nop_percentage = i / 100.0
                        output_folder = os.path.join(output_folder_base, str(i))
                        output_subfolder = os.path.join(output_folder, family_name)
                        os.makedirs(output_subfolder, exist_ok=True)

                        try:
                            # Determine the section and NOP code once
                            key = (file_name, "")
                            if key not in processed_files:
                                modified_data, section_name, nop_code_hex = inject_random_nop(file_path, 1.0)
                                processed_files[key] = (section_name, nop_code_hex)
                            else:
                                section_name, nop_code_hex = processed_files[key]

                            output_file_path = os.path.join(output_subfolder, f"modified_{i}_{file_name}")
                            
                            # Use the same NOP code for all percentages
                            modified_data, _, _ = inject_random_nop(file_path, nop_percentage, bytes.fromhex(nop_code_hex))

                            if (file_name, section_name, nop_code_hex) not in processed_files:
                                csv_writer.writerow([file_name, section_name, nop_code_hex, "Success", "None"])
                                processed_files[(file_name, section_name, nop_code_hex)] = ("Success", "None")

                            with open(output_file_path, 'wb') as f:
                                f.write(modified_data)

                            print(f"Modified {file_name} with {nop_percentage*100}% NOP code in {section_name} saved as {output_file_path}")

                        except Exception as e:
                            print(f"Error processing {file_name}: {e}")
                            csv_writer.writerow([file_name, "N/A", "N/A", "Failed", str(e)])


In [20]:
input_folder = "/media/doonu/H/Malware/"
output_folder_base = "/media/doonu/H/Problem_Space/Manipulated_Executable_NOP/"
csv_path = "/media/doonu/H/Problem_Space/Modified_sections_log.csv"
process_executables(input_folder, output_folder_base, csv_path)

Modified miner_banload_2019_1472.exe with 5.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Banload/modified_5_miner_banload_2019_1472.exe
Modified miner_banload_2019_1472.exe with 15.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Banload/modified_15_miner_banload_2019_1472.exe
Modified miner_banload_2019_1472.exe with 25.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Banload/modified_25_miner_banload_2019_1472.exe
Modified miner_banload_2019_1472.exe with 35.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Banload/modified_35_miner_banload_2019_1472.exe
Modified miner_banload_2019_1472.exe with 45.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Banload/modified_45_miner_banload_2019_1472.exe
Modified miner_banload_2019_1091.exe with 5.0% NOP code in .text saved as /media/doonu/H/Prob

Modified miner_banload_2019_1683.exe with 15.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Banload/modified_15_miner_banload_2019_1683.exe
Modified miner_banload_2019_1683.exe with 25.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Banload/modified_25_miner_banload_2019_1683.exe
Modified miner_banload_2019_1683.exe with 35.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Banload/modified_35_miner_banload_2019_1683.exe
Modified miner_banload_2019_1683.exe with 45.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Banload/modified_45_miner_banload_2019_1683.exe
Modified miner_banload_2019_863.exe with 5.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Banload/modified_5_miner_banload_2019_863.exe
Modified miner_banload_2019_863.exe with 15.0% NOP code in .text saved as /media/doonu/H/Proble

Modified miner_banload_2019_1216.exe with 5.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Banload/modified_5_miner_banload_2019_1216.exe
Modified miner_banload_2019_1216.exe with 15.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Banload/modified_15_miner_banload_2019_1216.exe
Modified miner_banload_2019_1216.exe with 25.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Banload/modified_25_miner_banload_2019_1216.exe
Modified miner_banload_2019_1216.exe with 35.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Banload/modified_35_miner_banload_2019_1216.exe
Modified miner_banload_2019_1216.exe with 45.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Banload/modified_45_miner_banload_2019_1216.exe
Modified miner_banload_2019_1552.exe with 5.0% NOP code in .text saved as /media/doonu/H/Prob

Modified miner_banload_2019_850.exe with 25.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Banload/modified_25_miner_banload_2019_850.exe
Modified miner_banload_2019_850.exe with 35.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Banload/modified_35_miner_banload_2019_850.exe
Modified miner_banload_2019_850.exe with 45.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Banload/modified_45_miner_banload_2019_850.exe
Modified miner_banload_2019_965.exe with 5.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Banload/modified_5_miner_banload_2019_965.exe
Modified miner_banload_2019_965.exe with 15.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Banload/modified_15_miner_banload_2019_965.exe
Modified miner_banload_2019_965.exe with 25.0% NOP code in .text saved as /media/doonu/H/Problem_Space/

Modified miner_banload_2019_1227.exe with 35.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Banload/modified_35_miner_banload_2019_1227.exe
Modified miner_banload_2019_1227.exe with 45.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Banload/modified_45_miner_banload_2019_1227.exe
Modified miner_banload_2019_1137.exe with 5.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Banload/modified_5_miner_banload_2019_1137.exe
Modified miner_banload_2019_1137.exe with 15.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Banload/modified_15_miner_banload_2019_1137.exe
Modified miner_banload_2019_1137.exe with 25.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Banload/modified_25_miner_banload_2019_1137.exe
Modified miner_banload_2019_1137.exe with 35.0% NOP code in .text saved as /media/doonu/H/Pro

...

Modified spyware_formbook_2017_8.exe with 5.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Formbook/modified_5_spyware_formbook_2017_8.exe
Modified spyware_formbook_2017_8.exe with 15.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Formbook/modified_15_spyware_formbook_2017_8.exe
Modified spyware_formbook_2017_8.exe with 25.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Formbook/modified_25_spyware_formbook_2017_8.exe
Modified spyware_formbook_2017_8.exe with 35.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Formbook/modified_35_spyware_formbook_2017_8.exe
Modified spyware_formbook_2017_8.exe with 45.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Formbook/modified_45_spyware_formbook_2017_8.exe
Modified spyware_formbook_2020_192.exe with 5.0% NOP code in .text saved as /media/doonu

Modified spyware_formbook_2021_279.exe with 5.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Formbook/modified_5_spyware_formbook_2021_279.exe
Modified spyware_formbook_2021_279.exe with 15.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Formbook/modified_15_spyware_formbook_2021_279.exe
Modified spyware_formbook_2021_279.exe with 25.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Formbook/modified_25_spyware_formbook_2021_279.exe
Modified spyware_formbook_2021_279.exe with 35.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Formbook/modified_35_spyware_formbook_2021_279.exe
Modified spyware_formbook_2021_279.exe with 45.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Formbook/modified_45_spyware_formbook_2021_279.exe
Error processing spyware_formbook_2021_265.exe: Injection size is ze

Error processing spyware_formbook_2021_76.exe: Not enough slack space in .text to inject 6 byte NOP code
Error processing spyware_formbook_2021_76.exe: Not enough slack space in .text to inject 6 byte NOP code
Modified spyware_formbook_2021_76.exe with 25.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Formbook/modified_25_spyware_formbook_2021_76.exe
Modified spyware_formbook_2021_76.exe with 35.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Formbook/modified_35_spyware_formbook_2021_76.exe
Modified spyware_formbook_2021_76.exe with 45.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Formbook/modified_45_spyware_formbook_2021_76.exe
Modified spyware_formbook_2020_31.exe with 5.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Formbook/modified_5_spyware_formbook_2020_31.exe
Modified spyware_formbook_2020_31.exe with 15.0% NOP 

Modified spyware_formbook_2021_80.exe with 5.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Formbook/modified_5_spyware_formbook_2021_80.exe
Modified spyware_formbook_2021_80.exe with 15.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Formbook/modified_15_spyware_formbook_2021_80.exe
Modified spyware_formbook_2021_80.exe with 25.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Formbook/modified_25_spyware_formbook_2021_80.exe
Modified spyware_formbook_2021_80.exe with 35.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Formbook/modified_35_spyware_formbook_2021_80.exe
Modified spyware_formbook_2021_80.exe with 45.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Formbook/modified_45_spyware_formbook_2021_80.exe
Modified spyware_formbook_2020_160.exe with 5.0% NOP code in .text saved as /m

Modified spyware_formbook_2021_87.exe with 5.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Formbook/modified_5_spyware_formbook_2021_87.exe
Modified spyware_formbook_2021_87.exe with 15.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Formbook/modified_15_spyware_formbook_2021_87.exe
Modified spyware_formbook_2021_87.exe with 25.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Formbook/modified_25_spyware_formbook_2021_87.exe
Modified spyware_formbook_2021_87.exe with 35.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Formbook/modified_35_spyware_formbook_2021_87.exe
Modified spyware_formbook_2021_87.exe with 45.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Formbook/modified_45_spyware_formbook_2021_87.exe
Modified spyware_formbook_2021_111.exe with 5.0% NOP code in .text saved as /m